# CardioAI — Preprocessing

**Milestone 2 → 3 bridge: Feature Engineering & Train/Test Split**

This notebook prepares the cleaned dataset for model training:
1. Reload the raw data (already verified clean in `01_EDA.ipynb`)
2. Separate nominal categorical features for one-hot encoding vs. features kept as-is
3. Stratified 80/20 train/test split
4. Fit a `StandardScaler` on numeric features (**fit on train only**, to avoid leakage)
5. Persist the fitted scaler and the processed splits for reuse in `03_Model_Training.ipynb`


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

RANDOM_STATE = 42

df = pd.read_csv("../data/CardioAI_Merged_Heart_Dataset.csv")
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


Loaded dataset: 599 rows x 14 columns


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52.0,1.0,0.0,125.0,212.0,0.0,1.0,168.0,0.0,1.0,2.0,2.0,3.0,0
1,53.0,1.0,0.0,140.0,203.0,1.0,0.0,155.0,1.0,3.1,0.0,0.0,3.0,0
2,70.0,1.0,0.0,145.0,174.0,0.0,1.0,125.0,1.0,2.6,0.0,0.0,3.0,0
3,61.0,1.0,0.0,148.0,203.0,0.0,1.0,161.0,0.0,0.0,2.0,1.0,3.0,0
4,62.0,0.0,0.0,138.0,294.0,1.0,1.0,106.0,0.0,1.9,1.0,3.0,2.0,0


## Feature Engineering

`cp` (chest pain type), `restecg` (resting ECG), `slope`, and `thal` are **nominal** categories (no inherent order), so they're one-hot encoded rather than treated as continuous numbers — otherwise the model would wrongly assume e.g. chest pain type 3 > chest pain type 1.

`sex`, `fbs`, `exang` are already binary (0/1) — no encoding needed.

`age`, `trestbps`, `chol`, `thalach`, `oldpeak`, and `ca` are kept as numeric / ordinal counts and scaled.


In [2]:
nominal_features = ["cp", "restecg", "slope", "thal"]
binary_features = ["sex", "fbs", "exang"]
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

df_encoded = pd.get_dummies(df, columns=nominal_features, prefix=nominal_features)

# Ensure clean boolean -> int conversion for the new dummy columns
dummy_cols = [c for c in df_encoded.columns if any(c.startswith(f"{f}_") for f in nominal_features)]
df_encoded[dummy_cols] = df_encoded[dummy_cols].astype(int)

print(f"Shape after one-hot encoding: {df_encoded.shape}")
df_encoded.head()


Shape after one-hot encoding: (599, 28)


,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,...,slope_0.0,slope_1.0,slope_2.0,slope_3.0,thal_0.0,thal_1.0,thal_2.0,thal_3.0,thal_6.0,thal_7.0
0,52.0,1.0,125.0,212.0,0.0,168.0,0.0,1.0,2.0,0,...,0,0,1,0,0,0,0,1,0,0
1,53.0,1.0,140.0,203.0,1.0,155.0,1.0,3.1,0.0,0,...,1,0,0,0,0,0,0,1,0,0
2,70.0,1.0,145.0,174.0,0.0,125.0,1.0,2.6,0.0,0,...,1,0,0,0,0,0,0,1,0,0
3,61.0,1.0,148.0,203.0,0.0,161.0,0.0,0.0,1.0,0,...,0,0,1,0,0,0,0,1,0,0
4,62.0,0.0,138.0,294.0,1.0,106.0,0.0,1.9,3.0,0,...,0,1,0,0,0,0,1,0,0,0


In [3]:
X = df_encoded.drop(columns=["target"])
y = df_encoded["target"]

print(f"Feature matrix: {X.shape}")
print(f"Target vector: {y.shape}")
print(f"\nFinal feature list ({len(X.columns)}):")
print(list(X.columns))


Feature matrix: (599, 27)
Target vector: (599,)

Final feature list (27):
['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca', 'cp_0.0', 'cp_1.0', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_0.0', 'restecg_1.0', 'restecg_2.0', 'slope_0.0', 'slope_1.0', 'slope_2.0', 'slope_3.0', 'thal_0.0', 'thal_1.0', 'thal_2.0', 'thal_3.0', 'thal_6.0', 'thal_7.0']


## Train / Test Split

80/20 split, **stratified on `target`** to preserve class balance in both sets. This split is fixed here and reused identically in the model-training notebook — the test set is not touched again until final evaluation.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]} rows ({y_train.mean():.1%} positive)")
print(f"Test set:  {X_test.shape[0]} rows ({y_test.mean():.1%} positive)")


Train set: 479 rows (50.3% positive)
Test set:  120 rows (50.0% positive)


## Scaling

`StandardScaler` is **fit on the training set only**, then applied to both train and test — this prevents test-set statistics from leaking into the training process.

Only the continuous/count features (`numeric_features`) are scaled; binary and one-hot columns are left as 0/1.

In [5]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

X_train_scaled.head()


,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,cp_0.0,...,slope_0.0,slope_1.0,slope_2.0,slope_3.0,thal_0.0,thal_1.0,thal_2.0,thal_3.0,thal_6.0,thal_7.0
466,0.293965,1.0,0.022914,-0.790670,0.0,0.793067,1.0,-0.916190,-0.722487,0,...,0,1,0,0,0,0,0,0,0,1
112,1.068250,0.0,2.683740,1.597778,0.0,0.182539,1.0,-0.916190,-0.722487,1,...,0,0,1,0,0,0,1,0,0,0
111,0.404577,0.0,2.129401,-0.426330,1.0,-0.166334,1.0,1.518030,1.356020,1,...,0,1,0,0,0,1,0,0,0,0
223,-0.037871,1.0,-0.365123,0.545242,0.0,0.095321,0.0,-0.481508,0.316766,0,...,1,0,0,0,0,0,1,0,0,0
388,-0.812156,1.0,0.355517,0.221384,0.0,0.269758,0.0,-0.916190,-0.722487,0,...,0,1,0,0,0,0,0,1,0,0


## Persist Processed Artifacts

Save the scaler and the processed splits so `03_Model_Training.ipynb` can load them directly without repeating this logic (and so the exact same preprocessing is guaranteed to be used for inference later in `app.py`).

In [6]:
import os
os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X.columns), "../models/feature_columns.pkl")

X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved: scaler.pkl, feature_columns.pkl")
print("Saved: processed train/test splits to data/processed/")


Saved: scaler.pkl, feature_columns.pkl
Saved: processed train/test splits to data/processed/


## Preprocessing Summary

| Step | Decision |
|---|---|
| Missing values | None present — no imputation needed |
| Nominal categoricals (`cp`, `restecg`, `slope`, `thal`) | One-hot encoded |
| Binary features (`sex`, `fbs`, `exang`) | Kept as-is (already 0/1) |
| Continuous/count features | `StandardScaler`, fit on train only |
| Split | 80/20, stratified on `target`, `random_state=42` |
| Leakage prevention | Scaler fit exclusively on training data; test set scaled using train-fitted parameters only |
| Persisted artifacts | `models/scaler.pkl`, `models/feature_columns.pkl`, `data/processed/{X_train, X_test, y_train, y_test}.csv` |

Ready for **Notebook 03: Model Training**.
